# Experiment 2: Fine-Tuning a Pre-trained LLM Using LoRA (Parameter-Efficient Fine-Tuning)

**Course:** HAAI302COM - Artificial Intelligence Laboratory

## 1. Aim

To fine-tune a pre-trained language model on a domain-specific dataset (text-to-SQL) using Low-Rank Adaptation (LoRA) through the PEFT library, and to evaluate the performance of the model before and after fine-tuning.

## 2. Objectives

- To understand why full fine-tuning of large language models is expensive.
- To understand the idea of low-rank adaptation and the role of the rank `r` and scaling `lora_alpha`.
- To attach LoRA adapters to a pre-trained model using the PEFT library.
- To fine-tune only the adapter parameters on a small instruction dataset.
- To compare the model's loss and generated output before and after fine-tuning.
- To save the trained adapter and reload it for inference.

## 3. Software Requirements

- Python 3.9 or later
- Google Colab (free T4 GPU) or Jupyter Notebook; a CPU is enough for the small GPT-2 model used here
- Internet connection for downloading the model and dataset
- Required Python libraries: `transformers`, `peft`, `datasets`, `accelerate`, `torch`, `pandas`
- For Falcon-7B on a GPU additionally: `bitsandbytes`

## 4. Theory

### A. Why parameter-efficient fine-tuning

A pre-trained model has millions or billions of weights. Updating all of them (full fine-tuning) needs memory for the weights, the gradients and the optimizer state, and produces a full-size copy of the model for every task. Parameter-efficient fine-tuning (PEFT) freezes the pre-trained weights and trains only a very small number of extra parameters.

### B. Low-Rank Adaptation (LoRA)

For a frozen weight matrix W of size d &times; k, LoRA learns an update as the product of two thin matrices: B (d &times; r) and A (r &times; k), where the rank r is much smaller than d and k. The layer output becomes

`output = W x + (lora_alpha / r) * B A x`

- A is initialised randomly and B with zeros, so training starts exactly from the pre-trained model.
- Only A and B are trained. For r = 16 this is typically well under 2 percent of the model's parameters.
- After training, B A can be merged into W, so inference has no extra cost. The adapter file is only a few megabytes.

### C. Key LoRA hyper-parameters

| Parameter | Description |
|---|---|
| `r` | Rank of the update matrices. Larger r gives more capacity and more trainable parameters. |
| `lora_alpha` | Scaling factor. The update is multiplied by `lora_alpha / r`. |
| `target_modules` | Which layers receive adapters (for GPT-2: the attention projections `c_attn` and `c_proj`). |
| `lora_dropout` | Dropout applied to the adapter input to reduce over-fitting. |
| `learning_rate` | LoRA tolerates a higher rate than full fine-tuning (about 1e-4 to 1e-3). |

### D. The task: text-to-SQL

The model receives a table definition and a question in English and must write the SQL query. A base model has seen SQL, but it does not know the exact prompt format, so it tends to continue the text instead of answering. Fine-tuning teaches the format and the style of answer.

## 5. Algorithm / Procedure

1. Install the required libraries and import them.
2. Load the text-to-SQL dataset and split it into training, validation and test sets.
3. Convert every example into a prompt (schema and question) followed by the SQL answer.
4. Tokenize the examples and mask the prompt tokens so that only the answer is learned.
5. Load the pre-trained model and evaluate it before fine-tuning.
6. Create a `LoraConfig` and wrap the model with `get_peft_model`.
7. Train the adapters with the Hugging Face `Trainer`.
8. Evaluate the fine-tuned model on the same test set.
9. Save the adapter, reload it and generate SQL for a new question.
10. Compare the results and record observations.

## 6. Python Program

### Step 1: Install the required libraries

In [1]:
!pip install -q transformers peft datasets accelerate torch pandas

### Step 2: Load the dataset and build prompts

The `sql-create-context` dataset has a table definition (`context`), an English `question` and the correct SQL `answer`. We use small slices so that the experiment also runs on a CPU.

In [2]:
import math, time, torch, pandas as pd
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForSeq2Seq, set_seed)
from peft import LoraConfig, get_peft_model, PeftModel

set_seed(42)
data = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=42)
train_raw = data.select(range(1000))
val_raw   = data.select(range(1000, 1100))
test_raw  = data.select(range(1100, 1140))

def build_prompt(e):
    return f"### Schema:\n{e['context']}\n### Question:\n{e['question']}\n### SQL:\n"

print(build_prompt(train_raw[0]) + train_raw[0]["answer"])

### Schema:
CREATE TABLE table_name_50 (venue VARCHAR, away_team VARCHAR)
### Question:
When Essendon played away; where did they play?
### SQL:
SELECT venue FROM table_name_50 WHERE away_team = "essendon"


### Step 3: Tokenize with label masking

For a causal model the labels are the input IDs. The prompt part is set to `-100` so that the loss is computed only on the SQL answer, and an end-of-sequence token is added so that the model learns to stop.

In [3]:
BASE_MODEL = "gpt2"   # on a GPU, use "tiiuae/falcon-7b" with target_modules=["query_key_value"] and 4-bit loading
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.eos_token

def tokenize(e):
    p = tok(build_prompt(e))["input_ids"]
    a = tok(e["answer"])["input_ids"] + [tok.eos_token_id]
    return {"input_ids": p + a, "attention_mask": [1] * (len(p) + len(a)), "labels": [-100] * len(p) + a}

train_ds = train_raw.map(tokenize, remove_columns=train_raw.column_names)
val_ds   = val_raw.map(tokenize, remove_columns=val_raw.column_names)
print(train_ds)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})


### Step 4: Load the pre-trained model and evaluate it before fine-tuning

`generate_sql` writes the SQL for each test question with greedy decoding and keeps the first line. A prediction counts as an exact match if it equals the correct query after normalising case, spaces and the final semicolon.

In [4]:
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
print(f"Parameters: {model.num_parameters() / 1e6:.1f} M")

norm = lambda s: " ".join(s.strip().rstrip(";").lower().split())

@torch.no_grad()
def generate_sql(m, examples):
    m.eval(); tok.padding_side = "left"; outs = []
    for i in range(0, len(examples), 8):
        batch = examples.select(range(i, min(i + 8, len(examples))))
        enc = tok([build_prompt(e) for e in batch], return_tensors="pt", padding=True)
        gen = m.generate(**enc, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
        outs += [tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0] for g in gen]
    return outs

def exact_match(preds, examples):
    return round(100 * sum(norm(p) == norm(g) for p, g in zip(preds, examples["answer"])) / len(preds), 1)

preds_before = generate_sql(model, test_raw)
print("Exact match before fine-tuning:", exact_match(preds_before, test_raw), "%")

Parameters: 124.4 M


Exact match before fine-tuning: 0.0 %


### Step 5: Attach LoRA adapters

`LoraConfig` describes the adapters. `get_peft_model` freezes the original weights and inserts the small trainable matrices.

In [5]:
lora_config = LoraConfig(
    r=16,                              # rank of the update matrices
    lora_alpha=32,                     # scaling factor (alpha / r = 2)
    target_modules=["c_attn", "c_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    fan_in_fan_out=True,               # GPT-2 stores its weights transposed
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,622,016 || all params: 126,061,824 || trainable%: 1.2867


### Step 6: Train the adapters

In [6]:
args = TrainingArguments(
    output_dir="lora-checkpoints", num_train_epochs=1,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=1e-3, lr_scheduler_type="cosine", warmup_steps=5,
    logging_steps=10, save_strategy="no", report_to="none", disable_tqdm=True,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))

loss_before = trainer.evaluate()["eval_loss"]
t0 = time.time()
trainer.train()
loss_after = trainer.evaluate()["eval_loss"]
print(f"Training time: {(time.time() - t0) / 60:.1f} min")
print(f"Validation loss: {loss_before:.3f} -> {loss_after:.3f}")

{'loss': '5.923', 'grad_norm': '5.235', 'learning_rate': '0.0009883', 'epoch': '0.16'}


{'loss': '2.147', 'grad_norm': '3.343', 'learning_rate': '0.000863', 'epoch': '0.32'}


{'loss': '1.079', 'grad_norm': '4.222', 'learning_rate': '0.0006338', 'epoch': '0.48'}


{'loss': '0.8885', 'grad_norm': '1.304', 'learning_rate': '0.0003662', 'epoch': '0.64'}


{'loss': '0.839', 'grad_norm': '1.673', 'learning_rate': '0.000137', 'epoch': '0.8'}


{'loss': '0.8009', 'grad_norm': '3.552', 'learning_rate': '1.169e-05', 'epoch': '0.96'}


{'train_runtime': '765.4', 'train_samples_per_second': '1.307', 'train_steps_per_second': '0.082', 'train_loss': '1.897', 'epoch': '1'}


### Step 7: Evaluate after fine-tuning

In [7]:
preds_after = generate_sql(model, test_raw)
print("Exact match after fine-tuning:", exact_match(preds_after, test_raw), "%")

Exact match after fine-tuning: 5.0 %


### Step 8: Save the adapter and reload it for inference

Only the adapter weights are saved, not the whole model.

In [8]:
import os
model.save_pretrained("lora-sql-adapter")
size_kb = os.path.getsize("lora-sql-adapter/adapter_model.safetensors") / 1024
print(f"Adapter file size: {size_kb:,.0f} KB")

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
tuned = PeftModel.from_pretrained(base, "lora-sql-adapter")

new = {"context": "CREATE TABLE students (id INTEGER, name VARCHAR, marks INTEGER)",
       "question": "List the names of students who scored more than 80 marks"}
enc = tok(build_prompt(new), return_tensors="pt")
out = tuned.generate(**enc, max_new_tokens=40, do_sample=False, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().split("\n")[0])

Adapter file size: 6,345 KB


SELECT id FROM students WHERE marks = "80" AND marks > 80


## 7. Experimental Observations

### Observation Table 1: LoRA configuration and trainable parameters

In [9]:
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
pd.DataFrame({"Base model": [BASE_MODEL], "Rank r": [16], "lora_alpha": [32], "Target modules": ["c_attn, c_proj"],
              "Total parameters": [f"{total:,}"], "Trainable parameters": [f"{trainable:,}"],
              "Trainable %": [round(100 * trainable / total, 2)]}).T.rename(columns={0: "Value"})

,Value
Base model,gpt2
Rank r,16
lora_alpha,32
Target modules,"c_attn, c_proj"
Total parameters,"126,061,824"
Trainable parameters,"1,622,016"
Trainable %,1.29


### Observation Table 2: Before and after fine-tuning

In [10]:
pd.DataFrame({"Metric": ["Validation loss", "Perplexity", "Exact match on test set (%)"],
              "Before fine-tuning": [round(loss_before, 3), round(math.exp(loss_before), 2), exact_match(preds_before, test_raw)],
              "After fine-tuning": [round(loss_after, 3), round(math.exp(loss_after), 2), exact_match(preds_after, test_raw)]}).set_index("Metric")

,Before fine-tuning,After fine-tuning
Metric,,
Validation loss,4.971,0.603
Perplexity,144.230,1.830
Exact match on test set (%),0.000,5.000


### Observation Table 3: Sample generated queries

In [11]:
pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 200)
pd.DataFrame({"Question": test_raw["question"][:4], "Correct SQL": test_raw["answer"][:4],
              "Before": preds_before[:4], "After": preds_after[:4]})

,Question,Correct SQL,Before,After
0,What is the score vs. Terran when the score vs. Zerg is 69?,"SELECT vs_terran FROM table_name_12 WHERE vs_zerg = ""69""",SELECT * FROM table_name_12 WHERE v = 0;,"SELECT score FROM table_name_12 WHERE vs_terran = 69 AND zerg = ""69"""
1,Where was the game played where Collingwood was the away team?,"SELECT venue FROM table_name_99 WHERE away_team = ""collingwood""",SELECT * FROM table_name_99 WHERE team_id = 'VFL',"SELECTvenue FROM table_name_99 WHERE away_team = ""Collingwood"""
2,What is every construction date for the registration of HB-HOS?,SELECT construction AS date FROM table_22180353_1 WHERE registrati...,"CREATE TABLE table_22180353_1 (construction VARCHAR, registration ...","SELECT construction FROM table_22180353_1 WHERE registration = ""HB..."
3,What is the highest effic with an avg/g of 91.9?,SELECT MAX(effic) FROM table_name_68 WHERE avg_g = 91.9,SELECT * FROM table_name_68 WHERE avg_g = 91.9,SELECT avg_g FROM table_name_68 WHERE avg_g = 91.9 AND avg_g < 91.9


**Observations**

- LoRA trained 1,622,016 of 126,061,824 parameters (1.29 percent). The saved adapter is about 6 MB, compared with roughly 500 MB for the full GPT-2 weights.
- Validation loss on the SQL tokens fell from 4.971 to 0.603, and perplexity from 144 to 1.8, after one pass over 1,000 examples (about 13 minutes on a CPU).
- Before fine-tuning the model never produced the requested query. It wrote generic `SELECT *` statements, invented a `team_id` column, or repeated a `CREATE TABLE` statement (exact match 0 percent).
- After fine-tuning the output follows the format: it reads the column names from the schema and writes a `SELECT ... FROM <table> WHERE ...` query. Exact match rose from 0 to 5 percent (2 of 40 test questions).
- Exact match is a strict measure for a small model. Many outputs are close but not identical, for example `"Collingwood"` instead of `"collingwood"`, or an extra `AND` condition.

## 8. Result

A GPT-2 model was fine-tuned for text-to-SQL using LoRA adapters (rank 16, alpha 32) through the PEFT library. Only 1.29 percent of the parameters were trained. The validation loss decreased from 4.971 to 0.603, the exact-match score on the test set increased from 0 percent to 5 percent, and the generated queries changed from generic or unrelated statements to queries that use the table and column names given in the schema. The trained adapter was saved (about 6 MB), reloaded on top of the base model and used to answer a new question.

## 9. Conclusion

LoRA makes it practical to adapt a pre-trained language model to a new task by training only a very small set of low-rank adapter matrices while the original weights stay frozen. Fine-tuning on a small text-to-SQL dataset taught the model the expected prompt format and the style of the answer, which reduced the validation loss and improved the quality of the generated queries. The adapter is small enough to store and share separately, and the same base model can serve many tasks by loading different adapters.